# DataHek OSS — 02 · The pipeline, step by step

The engine is deliberately explicit: **schema → plan → validate → guardrails
→ execute → mask → explain**. This notebook walks each step with real
objects, no API layer in the way.

In [1]:
import sys, os
for _c in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.join(os.path.dirname(os.getcwd()), "notebooks")):
    if os.path.exists(os.path.join(_c, "datahek_demo.py")):
        sys.path.insert(0, _c)
        break
import tempfile
os.environ.setdefault("DATAHEK_DB_PATH", os.path.join(tempfile.gettempdir(), "datahek_notebooks.db"))

import asyncio
from datahek_demo import build_demo_db, make_container

db = build_demo_db()
container = make_container(db)
print("container ready")

container ready


## 1 · Schema discovery
The provider introspects the database and returns a catalog.

In [2]:
from datahek.contracts.connections import ConnectionManager
from datahek.engine.executor import ProviderRegistry
from datahek.engine.schema import SchemaService
from datahek.kernel.context import RequestContext

ctx = RequestContext(source="notebook")
conn = (await container.resolve(ConnectionManager).list_connections(ctx))[0]
provider = container.resolve(ProviderRegistry).get(conn.provider)
catalog = await container.resolve(SchemaService).get_catalog(ctx, conn, provider)
print("tables:")
for t in catalog.tables:
    print(" -", t.name, "->", [c.name for c in t.columns])

tables:
 - service_meta -> ['service', 'tier', 'owner']
 - traces -> ['service', 'status', 'duration_ms', 'ts']


## 2 · Plan
The planner (an LLM in production, a stub here) proposes a **LogicalPlan**.

In [3]:
from datahek.engine.planner import Planner

plan_result = await container.resolve(Planner).plan(
    "What is the average duration per service?", ctx, conn, provider)
plan = plan_result.plan
print("sources:", plan_result.sources_used)
print("plan:", plan.to_dict()["nodes"][0])

sources: ['traces']
plan: {'type': 'ReadNode', 'source': 'traces', 'columns': ['service'], 'filter': None, 'group_by': ['service'], 'aggregates': [{'function': 'avg', 'column': 'duration_ms', 'alias': 'avg_duration'}], 'order_by': ['avg_duration DESC'], 'limit': 10, 'joins': []}


## 3 · Validate
Deterministic checks: tables, columns, dialect functions, read-only shape.

In [4]:
from datahek.engine.plan import validate_plan

tables = container.resolve(SchemaService).tables(catalog)
columns = container.resolve(SchemaService).columns(catalog)
validate_plan(plan, tables, columns, dialect=provider.capabilities.dialect)
print("plan is valid ✓")

plan is valid ✓


## 4 · Guardrails
The pipeline returns a typed decision — ALLOW / DENY / REQUIRE_APPROVAL / RATE_LIMIT.

In [5]:
from datahek.engine.executor import Engine

engine = container.resolve(Engine)
result = await engine.execute(ctx, plan, conn)
print("executed:", result.row_count, "rows")

executed: 4 rows


## 5 · What was compiled
Every execution compiles the plan to SQL — reproducible and auditable.

In [6]:
from datahek.engine.compile import compile_sql
print(compile_sql(plan))

SELECT service, avg(duration_ms) AS avg_duration FROM traces GROUP BY service ORDER BY avg_duration DESC LIMIT 10


## 6 · Explain
The reasoner formats the result for humans (an LLM in production).

In [7]:
from datahek.contracts.reasoner import Reasoner

explanation = await container.resolve(Reasoner).explain("avg per service", result, plan, ctx)
print(explanation)

Stub explanation: the result is shown in the table below.


**Takeaway:** the LLM only ever proposes; deterministic code decides and executes.